<div style="border-top:4px solid #0f766e;padding:28px 0 18px"><div style="color:#0f766e;font-weight:700;letter-spacing:.8px">模块 10：指标加工与服务交付</div><div style="color:#17212b;font-size:30px;font-weight:750">模块 10：指标加工与服务交付</div><p style="color:#475569;line-height:1.7">在声明的粒度上定义指标，构建可复用的聚合结果，并为下游看板提供稳定查询。请按顺序运行；结果会以表格展示，写入只作用于本模块的 `_l2` 对象。</p></div>

## 边界

本实验不会修改 Level 1 源表，只创建或替换带 `_l2` 后缀的对象。

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox

lab = connect_sandbox()


In [ ]:
lab.execute("DROP TABLE IF EXISTS daily_order_metrics_l2")
lab.execute("""
CREATE TABLE daily_order_metrics_l2 (
    order_date DATE NOT NULL,
    data_source VARCHAR(32) NOT NULL,
    order_count BIGINT SUM NOT NULL DEFAULT "0",
    gross_amount DECIMAL(18,2) SUM NOT NULL DEFAULT "0.00"
)
AGGREGATE KEY(order_date, data_source)
DISTRIBUTED BY HASH(order_date) BUCKETS 1
PROPERTIES ("replication_num"="1")
""")
lab.execute("""
INSERT INTO daily_order_metrics_l2
SELECT DATE(event_time), data_source, COUNT(*), SUM(order_amount)
FROM orders_imported
GROUP BY DATE(event_time), data_source
""")
lab.sql("SELECT * FROM daily_order_metrics_l2 ORDER BY order_date, data_source", title="声明粒度的服务表")

In [ ]:
lab.sql("""
SELECT order_date,
       SUM(CASE WHEN data_source = 'COURSE_SIMULATION' THEN order_count ELSE 0 END) AS simulated_orders,
       SUM(order_count) AS all_orders,
       SUM(gross_amount) AS gross_amount
FROM daily_order_metrics_l2
GROUP BY order_date
HAVING SUM(order_count) > 0
ORDER BY order_date
""", title="看板指标查询")

In [ ]:
lab.sql("""
WITH detail AS (
    SELECT DATE(event_time) AS order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
    FROM orders_imported
    GROUP BY DATE(event_time)
), serving AS (
    SELECT order_date, SUM(order_count) AS order_count, SUM(gross_amount) AS gross_amount
    FROM daily_order_metrics_l2
    GROUP BY order_date
)
SELECT d.order_date,
       d.order_count AS detail_orders,
       s.order_count AS serving_orders,
       d.gross_amount AS detail_amount,
       s.gross_amount AS serving_amount
FROM detail AS d
JOIN serving AS s ON s.order_date = d.order_date
ORDER BY d.order_date
""", title="明细与服务表的独立核对")

## 要点

将结果与课程中声明的业务粒度对照。SQL 执行成功本身不能证明模型、指标、访问边界或消费者契约正确。